# Day 3 -- Convolutional Neural Networks: Giving the Network Eyes

Every network so far has started with `Flatten()` -- turning a 28x28 image into a flat list of
784 numbers before the first Dense layer even sees it. That throws away something important:
*which pixels are next to which other pixels*. Today we fix that with **convolutional layers**,
which look at small neighborhoods of pixels at a time, the way your own eyes scan an image for
edges and shapes rather than processing every pixel in isolation.

## Hook (10 min)

Run the cell below. It takes one Fashion-MNIST image, shuffles its pixels into a random order,
and displays both versions side by side.

In [5]:
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

# load_data() returns two pairs: (training images, training labels) and (test images, test labels).
# Unpacking them on separate lines (instead of one nested line) keeps each step easy to follow.
train_data, test_data = keras.datasets.fashion_mnist.load_data()
x_train, y_train = train_data
x_test, y_test = test_data

sample = x_train[1]
rng = np.random.default_rng(seed=0)
shuffled = sample.flatten().copy()
rng.shuffle(shuffled)
shuffled = shuffled.reshape(28, 28)

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(sample, cmap="gray"); axes[0].set_title(f"Original ({class_names[y_train[1]]})"); axes[0].axis("off")
axes[1].imshow(shuffled, cmap="gray"); axes[1].set_title("Pixels shuffled"); axes[1].axis("off")
plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'tensorflow'

**Discussion**: You can't recognize the shuffled image, but it contains the *exact same 784
pixel values* as the original -- just rearranged. A plain `Dense` layer, working on a flattened
vector, has no concept of "next to" -- it would treat both images with equal ease (or equal
difficulty). It's only using neighborhood/spatial structure that makes an image recognizable.
That's exactly what convolution is built to exploit.

## Concept (30 min)

### The convolution operation

A **filter** (also called a kernel) is a small grid of numbers -- commonly 3x3. It slides across
the image, and at each position it computes a single weighted sum of the pixels underneath it
(multiply-and-add, just like the single neuron from Day 1, but applied locally instead of to the
whole image at once). The output is a new, filtered image called a **feature map**.

Different filters detect different things. A filter with large positive weights on one side and
large negative weights on the other detects an **edge** at that boundary. We'll see this live
below before letting Keras learn its own filters.

### Feature maps and learned filters

A `Conv2D` layer in Keras doesn't use one hand-designed filter -- it learns many filters
(commonly 32 or 64) simultaneously, each producing its own feature map, during training via the
exact same backpropagation process from Day 2. Early layers tend to learn simple filters (edges,
color blobs, textures); layers deeper in the network combine those into filters for more complex
shapes.

### Pooling

A **MaxPooling** layer downsamples a feature map -- e.g. `MaxPooling2D((2,2))` slides a 2x2
window across the feature map and keeps only the largest value in each window, halving both
height and width. This does two things: reduces the amount of computation needed downstream, and
makes the network slightly more tolerant of an object shifting by a pixel or two (the strongest
signal in a neighborhood survives pooling even if its exact position shifts a little).

### A typical CNN pattern

```
Conv2D -> MaxPooling2D -> Conv2D -> MaxPooling2D -> Flatten -> Dense -> Dense(softmax)
```

Notice `Flatten` still shows up -- but now only at the *end*, after the convolutional layers
have already extracted spatially-aware features. The final Dense layers just combine those
already-meaningful features into a classification decision.

## Part A -- Instructor-Led Demo

### Guided Demo, Part 1 (45 min) -- Manual convolution, then a real CNN

In [ ]:
# Before letting Keras learn filters, apply one by hand so the operation itself is concrete.
from scipy.signal import convolve2d

# A simple vertical-edge detector.
edge_kernel = np.array([
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1],
])

sample_image = x_train[1].astype("float32")
filtered = convolve2d(sample_image, edge_kernel, mode="valid")

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(sample_image, cmap="gray"); axes[0].set_title("Original"); axes[0].axis("off")
axes[1].imshow(filtered, cmap="gray"); axes[1].set_title("After one hand-built 3x3 filter"); axes[1].axis("off")
plt.tight_layout()
plt.show()

print("Original shape:", sample_image.shape, "-> Filtered shape:", filtered.shape)
print("Notice the filtered image is smaller -- a 3x3 filter can't be centered on the outermost pixels.")

NameError: name 'np' is not defined

That's a single hand-designed filter. A `Conv2D` layer does exactly this operation, but learns
the filter's numbers from data instead of us choosing them, and learns dozens of them at once.

In [ ]:
# CNNs in Keras expect an explicit channel dimension, even for grayscale images: (28, 28, 1).
# np.expand_dims adds a new axis at the end (axis=-1), turning shape (28, 28) into (28, 28, 1).
x_train_norm = np.expand_dims(x_train.astype("float32") / 255.0, axis=-1)
x_test_norm = np.expand_dims(x_test.astype("float32") / 255.0, axis=-1)
print("Input shape for the CNN:", x_train_norm.shape)

Input shape for the CNN: (60000, 28, 28, 1)


In [ ]:
# CNN Architecture
cnn_model = keras.Sequential([
    keras.layers.Conv2D(32, (3, 3), activation="relu", input_shape=(28, 28, 1)),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64, (3, 3), activation="relu"),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(10, activation="softmax"),
])

cnn_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cnn_model.summary()

NameError: name 'keras' is not defined

Compare the parameter count here to Day 1's Dense-only model. Despite "seeing" the same size
image, a CNN this deep often has *fewer* parameters than a single big Dense layer, because each
filter's weights are reused (shared) across every position in the image, instead of every input
pixel needing its own unique weight to every neuron.

**Break (10 min)**

### Guided Demo, Part 2 (20 min) -- Train, compare, and visualize

In [ ]:
early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

cnn_history = cnn_model.fit(
    x_train_norm, y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1,
)

Epoch 1/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 48s 55ms/step - accuracy: 0.7701 - loss: 0.6320 - val_accuracy: 0.8465 - val_loss: 0.4078
Epoch 2/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 45s 53ms/step - accuracy: 0.8486 - loss: 0.4205 - val_accuracy: 0.8770 - val_loss: 0.3375
Epoch 3/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 44s 53ms/step - accuracy: 0.8685 - loss: 0.3661 - val_accuracy: 0.8860 - val_loss: 0.3116
Epoch 4/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 84s 55ms/step - accuracy: 0.8802 - loss: 0.3342 - val_accuracy: 0.8865 - val_loss: 0.2993
Epoch 5/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 80s 53ms/step - accuracy: 0.8872 - loss: 0.3096 - val_accuracy: 0.8993 - val_loss: 0.2734
Epoch 6/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 44s 53ms/step - accuracy: 0.8949 - loss: 0.2903 - val_accuracy: 0.9013 - val_loss: 0.2734
Epoch 7/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 84s 55ms/step - accuracy: 0.8999 - loss: 0.2746 - val_accuracy: 0.9043 - val_loss: 0.2578
Epoch 8/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 44s 53ms/step - accuracy: 0.9049 - loss: 0.2570 - 

In [ ]:
cnn_test_loss, cnn_test_acc = cnn_model.evaluate(x_test_norm, y_test, verbose=0)
print(f"CNN test accuracy: {cnn_test_acc:.4f}")
print("Compare this to Day 2's best Dense (Dropout + EarlyStopping) model's test accuracy --")
print("the CNN should come out ahead on this image task, using a similar or smaller parameter budget.")

In [ ]:
# Visualize what the FIRST conv layer's learned filters actually look like.
first_conv_weights = cnn_model.layers[0].get_weights()[0]   # shape: (3, 3, 1, 32)
fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(first_conv_weights[:, :, 0, i], cmap="gray")
    ax.axis("off")
plt.suptitle("All 32 learned 3x3 filters from the first Conv2D layer")
plt.tight_layout()
plt.show()

In [ ]:
# Visualize a FEATURE MAP: what the first conv layer's output looks like for one real image.
feature_extractor = keras.Model(inputs=cnn_model.inputs, outputs=cnn_model.layers[0].output)
sample_batch = x_test_norm[0:1]
feature_maps = feature_extractor.predict(sample_batch, verbose=0)[0]   # shape: (26, 26, 32)

fig, axes = plt.subplots(1, 6, figsize=(14, 2.5))
axes[0].imshow(x_test[0], cmap="gray"); axes[0].set_title("Input"); axes[0].axis("off")
for i in range(1, 6):
    axes[i].imshow(feature_maps[:, :, i - 1], cmap="viridis")
    axes[i].set_title(f"Filter {i}"); axes[i].axis("off")
plt.suptitle("Input image and 5 of its feature maps from the first Conv2D layer")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

y_pred = np.argmax(cnn_model.predict(x_test_norm, verbose=0), axis=1)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xticks(rotation=45, ha="right")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("CNN Confusion Matrix (Fashion-MNIST)")
plt.tight_layout()
plt.show()

Look at which classes get confused most (commonly Shirt vs. T-shirt/top vs. Pullover -- they
really do look alike at 28x28 resolution). That's a genuinely harder confusion than anything in
Day 1's digit classifier, which is exactly why Fashion-MNIST is a better dataset for this kind
of diagnostic exercise.

## Part B -- Your Turn (Short Assignment, ~30-40 min)

In [ ]:
# TODO 1: Add a third Conv2D + MaxPooling2D block.
# Same architecture as the demo model, but with a THIRD Conv2D(128, (3,3), activation="relu")
# layer inserted just before Flatten() -- no MaxPooling after it, since by that point the
# feature map is already small (5x5 -> 3x3 after this conv).
three_block_model = keras.Sequential([
    keras.layers.Conv2D(32, (3, 3), activation="relu", input_shape=(28, 28, 1)),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64, (3, 3), activation="relu"),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(128, (3, 3), activation="relu"),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(10, activation="softmax"),
])

three_block_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
three_block_model.summary()

# Same EarlyStopping settings as the demo model.
three_block_early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

three_block_history = three_block_model.fit(
    x_train_norm, y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    callbacks=[three_block_early_stop],
    verbose=1,
)

three_block_test_loss, three_block_test_acc = three_block_model.evaluate(x_test_norm, y_test, verbose=0)

print(f"2-block demo   -> test accuracy: {cnn_test_acc:.4f} | params: {cnn_model.count_params():,}")
print(f"3-block (TODO1)-> test accuracy: {three_block_test_acc:.4f} | params: {three_block_model.count_params():,}")

**Your answer (TODO 1)**: Did the third conv block improve test accuracy? Was it worth the
added parameters and training time?

No -- the third conv block actually made things slightly *worse*, not better. The 2-block demo
model reached **91.16% test accuracy with 121,930 params**, while the 3-block model reached
**90.98% with 167,114 params** (~37% more parameters). EarlyStopping also triggered sooner for
the 3-block model (12 epochs vs. the full 15), suggesting the extra capacity started overfitting
before it found any genuinely new features to extract. On a small, low-resolution dataset like
Fashion-MNIST, two conv blocks already capture what there is to capture at 28x28 -- the third
block wasn't worth its added parameters or training cost here.

In [ ]:
# TODO 2: Change the first Conv2D layer's filter count.
# Rebuild the original 2-block demo CNN, but with 16 filters instead of 32 in the first Conv2D
# layer. Everything else (second conv layer, dense head, EarlyStopping) stays identical so the
# filter count is the only thing that changes.
small_filter_model = keras.Sequential([
    keras.layers.Conv2D(16, (3, 3), activation="relu", input_shape=(28, 28, 1)),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64, (3, 3), activation="relu"),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(10, activation="softmax"),
])

small_filter_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
small_filter_model.summary()

small_filter_early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

small_filter_history = small_filter_model.fit(
    x_train_norm, y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    callbacks=[small_filter_early_stop],
    verbose=1,
)

small_filter_test_loss, small_filter_test_acc = small_filter_model.evaluate(x_test_norm, y_test, verbose=0)

print(f"32-filter demo -> test accuracy: {cnn_test_acc:.4f} | params: {cnn_model.count_params():,}")
print(f"16-filter (TODO2) -> test accuracy: {small_filter_test_acc:.4f} | params: {small_filter_model.count_params():,}")

**Your answer (TODO 2)**: What effect did halving the filter count have on accuracy and model
size?

Halving the first layer's filters (32 -> 16) dropped test accuracy from **91.16% to 90.08%**
(about 1.1 percentage points), while only shrinking the model from **121,930 to 112,554 params**
(~7.7% smaller). The parameter savings look small because the cut cascades: the second Conv2D
layer's weights also shrink, since it now reads only 16 input channels instead of 32. So this
trade was worse than the 3-block experiment in TODO 1 -- a bigger accuracy hit for a smaller
parameter saving -- which suggests the first layer's filter count (how many distinct edge/texture
detectors it learns) matters more to this model's accuracy than adding a third conv block did.

---

## What's next

Today's CNN was trained from scratch on 60,000 images. Tomorrow: what do you do when you only
have a few hundred images? **Transfer learning** -- reusing a network someone else already
trained on millions of images.